In [1]:
#!pip install -q -U scikit-learn pandas numpy joblib
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


## Загрузка датасета

In [2]:
from pathlib import Path
import random
import zipfile

import joblib
import numpy as np
import pandas as pd
import catboost

from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
ARTIFACT_DIR = Path("artifacts")

RAW_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print(catboost.__version__)

1.2.10


In [4]:
!wget -q -O data/raw/bank-additional.zip \
  https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip

!unzip -qo data/raw/bank-additional.zip -d data/raw/

In [5]:
DATA_PATH = RAW_DIR / "bank-additional" / "bank-additional-full.csv"

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Shape: {df.shape}")
df.head()

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [6]:
list(df.columns)

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

## Предобработка датасета

In [7]:
print(f"Shape: {df.shape}")
display(df.head())

print("Типы колонок:")
display(df.dtypes)

print("Пропуски:")
display(df.isna().sum().sort_values(ascending=False))

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


Типы колонок:


,0
age,int64
job,object
marital,object
education,object
default,object
housing,object
loan,object
contact,object
month,object
day_of_week,object


Пропуски:


,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


Пропусков нет, целевая переменная несбалансированна. В данном датасете также есть leakage - колонка duration, которую надо дропнуть.

In [8]:
target = 'y'
df[target] = (
    df[target]
    .map({"no": 0, "yes": 1})
    .astype("int8")
)

print(df[target].value_counts())
print(f"\nДоля положительного класса: {df[target].mean():.2%}")

y
0    36548
1     4640
Name: count, dtype: int64

Доля положительного класса: 11.27%


In [9]:
leakage_col = 'duration'
df.drop(columns = leakage_col, inplace = True)

In [10]:
X = df.drop(columns = [target])
y = df[target]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_val,
)

In [12]:
categorical_features = X_train.select_dtypes(
    include = ["object", "category"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include = ["number", "bool"]
).columns.tolist()

print("Категориальные признаки:")
print(categorical_features)

print("Числовые признаки:")
print(numeric_features)

Категориальные признаки:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Числовые признаки:
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [15]:
def prepare_features_for_catboost(dataframe: pd.DataFrame, numeric_medians: dict) -> pd.DataFrame:
    result = dataframe.copy()

    for column in categorical_features:
        result[column] = result[column].fillna("__MISSING__").astype(str)

    for column in numeric_features:
        result[column] = pd.to_numeric(result[column], errors="coerce")
        result[column] = result[column].fillna(numeric_medians[column])

    return result

In [16]:
numeric_medians = X_train[numeric_features].median().to_dict()
print("Медианы числовых колонок: ", numeric_medians)

X_train_cb = prepare_features_for_catboost(X_train, numeric_medians)
X_val_cb = prepare_features_for_catboost(X_val, numeric_medians)
X_test_cb = prepare_features_for_catboost(X_test, numeric_medians)

X_train_cb.head()

Медианы числовых колонок:  {'age': 38.0, 'campaign': 2.0, 'pdays': 999.0, 'previous': 0.0, 'emp.var.rate': 1.1, 'cons.price.idx': 93.749, 'cons.conf.idx': -41.8, 'euribor3m': 4.857, 'nr.employed': 5191.0}


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
29286,40,housemaid,married,basic.6y,unknown,unknown,unknown,cellular,apr,fri,2,999,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1
1679,39,services,married,high.school,no,yes,no,telephone,may,fri,2,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0
38939,25,technician,single,professional.course,no,yes,no,cellular,nov,wed,1,999,0,nonexistent,-3.4,92.649,-30.1,0.716,5017.5
12809,25,technician,single,university.degree,no,no,no,cellular,jul,tue,3,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1
23180,41,unemployed,divorced,professional.course,no,unknown,unknown,cellular,aug,tue,3,999,0,nonexistent,1.4,93.444,-36.1,4.965,5228.1


## Обучение CatBoost

In [17]:
catboost_model = CatBoostClassifier(
    iterations=1_000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=100,
    od_type="Iter",
    od_wait=100,
    use_best_model=True,
    task_type="CPU",
    allow_writing_files=False,
)


catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val),
)

0:	learn: 0.7820982	test: 0.7823406	best: 0.7823406 (0)	total: 117ms	remaining: 1m 57s
100:	learn: 0.8254521	test: 0.8150797	best: 0.8155770 (76)	total: 5.83s	remaining: 51.9s
200:	learn: 0.8340223	test: 0.8176346	best: 0.8178306 (197)	total: 13s	remaining: 51.8s
300:	learn: 0.8509366	test: 0.8220135	best: 0.8220135 (300)	total: 19.7s	remaining: 45.7s
400:	learn: 0.8613326	test: 0.8236406	best: 0.8236534 (399)	total: 28.1s	remaining: 41.9s
500:	learn: 0.8699540	test: 0.8243281	best: 0.8246115 (495)	total: 35.5s	remaining: 35.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8246115377
bestIteration = 495

Shrink model to first 496 iterations.


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, eval_metric='PRAUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', od_type='Iter', od_wait=100, random_seed=42, task_type='CPU', use_best_model=True, verbose=100)

In [18]:
print(f"Лучшая итерация: {catboost_model.get_best_iteration()}")
print(f"Количество деревьев: {catboost_model.tree_count_}")

Лучшая итерация: 495
Количество деревьев: 496


## Оценка Модели

In [19]:
def evaluate_classifier(
    model,
    X_data: pd.DataFrame,
    y_true: pd.Series,
    threshold: float = 0.5,
    dataset_name: str = "validation",
):
    y_proba = model.predict_proba(X_data)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "dataset": dataset_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    display(pd.DataFrame([metrics]).round(4))

    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return metrics, y_proba, y_pred


val_metrics_05, val_proba, val_pred_05 = evaluate_classifier(
    model=catboost_model,
    X_data=X_val_cb,
    y_true=y_val,
    threshold=0.5,
    dataset_name="validation"
    )

,dataset,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,validation,0.5,0.8059,0.4651,0.8393,0.3739,0.6334,0.4702


Classification report:
              precision    recall  f1-score   support

           0     0.9490    0.8654    0.9053      5848
           1     0.3739    0.6334    0.4702       742

    accuracy                         0.8393      6590
   macro avg     0.6615    0.7494    0.6878      6590
weighted avg     0.8842    0.8393    0.8563      6590

Confusion matrix:
[[5061  787]
 [ 272  470]]


PR-AUC равен 0.4651. Так как доля положительного класса равна 11%, то у случайного предсказания PR-AUC был бы равен 0.11. То есть результат нашей модели намного лучше рандома, несмотря на сравнительно небольшое значение самой метрики.

Сделаем также подбор лучшей границы. Делать его будем через оптимизацию F1-score.

In [20]:
thresholds = np.arange(0.05, 0.96, 0.01)

results = []

for threshold in thresholds:
    predictions = (val_proba >= threshold).astype(int)

    results.append(
        {
            "threshold": float(threshold),
            "precision": precision_score(y_val, predictions, zero_division=0),
            "recall": recall_score(y_val, predictions, zero_division=0),
            "f1": f1_score(y_val, predictions, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(results)

best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
BEST_THRESHOLD = float(best_threshold_row["threshold"])

print(f"Лучший threshold по F1: {BEST_THRESHOLD:.2f}")
display(pd.DataFrame([best_threshold_row]).round(4))

print("\nТоп-10 порогов по F1:")
display(threshold_df.sort_values("f1", ascending=False).head(10))

Лучший threshold по F1: 0.71


,threshold,precision,recall,f1
66,0.71,0.4809,0.5606,0.5177



Топ-10 порогов по F1:


,threshold,precision,recall,f1
66,0.71,0.480925,0.560647,0.517735
67,0.72,0.484099,0.553908,0.516656
68,0.73,0.487864,0.541779,0.513410
65,0.70,0.470588,0.560647,0.511685
69,0.74,0.491803,0.525606,0.508143
64,0.69,0.462819,0.561995,0.507608
70,0.75,0.496757,0.516173,0.506279
63,0.68,0.457831,0.563342,0.505136
61,0.66,0.449001,0.575472,0.504430
60,0.65,0.446058,0.579515,0.504103


Видим, что лучшая F1 получилась при границе, равной 0.71.

In [21]:
test_metrics, test_proba, test_pred = evaluate_classifier(
    model=catboost_model,
    X_data=X_test_cb,
    y_true=y_test,
    threshold=BEST_THRESHOLD,
    dataset_name="test",
)

,dataset,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,test,0.71,0.813,0.4878,0.8849,0.4908,0.5722,0.5284


Classification report:
              precision    recall  f1-score   support

           0     0.9445    0.9246    0.9345      7310
           1     0.4908    0.5722    0.5284       928

    accuracy                         0.8849      8238
   macro avg     0.7176    0.7484    0.7314      8238
weighted avg     0.8934    0.8849    0.8887      8238

Confusion matrix:
[[6759  551]
 [ 397  531]]


## Важности фичей

In [22]:
feature_importance = pd.DataFrame(
    {
        "feature": X_train_cb.columns,
        "importance": catboost_model.get_feature_importance(),
    }
).sort_values("importance", ascending=False)

display(feature_importance.head(20))

,feature,importance
17,euribor3m,16.670228
18,nr.employed,8.411331
8,month,8.152938
10,campaign,7.776384
1,job,7.704804
3,education,7.247387
0,age,6.173012
14,emp.var.rate,4.714561
16,cons.conf.idx,4.551202
9,day_of_week,4.482667


Попробуем сузить число необходимых фичей для упрощения продакшена, избегая больших потерь в качестве. Для этого рассмотрим PR-AUC в зависимости от числа используемых самых важных фичей.

In [23]:
from sklearn.metrics import average_precision_score

FEATURE_STEP = 2

# Признаки в порядке убывания важности
ranked_features = feature_importance["feature"].tolist()

results = []

for n_features in range(FEATURE_STEP, len(ranked_features) + 1, FEATURE_STEP):
    selected_features = ranked_features[:n_features]

    X_train_selected = X_train_cb[selected_features]
    X_val_selected = X_val_cb[selected_features]

    # Только категориальные признаки, которые вошли в top-N
    selected_cat_features = [
        feature
        for feature in categorical_features
        if feature in selected_features
    ]

    model = CatBoostClassifier(
        iterations=1_000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        random_seed=SEED,
        verbose=False,
        od_type="Iter",
        od_wait=100,
        use_best_model=True,
        task_type="CPU",
        allow_writing_files=False,
    )

    model.fit(
        X_train_selected,
        y_train,
        cat_features=selected_cat_features,
        eval_set=(X_val_selected, y_val),
    )

    val_proba = model.predict_proba(X_val_selected)[:, 1]
    pr_auc = average_precision_score(y_val, val_proba)

    results.append(
        {
            "n_features": n_features,
            "pr_auc": pr_auc,
            "best_iteration": model.get_best_iteration(),
            "features": selected_features,
        }
    )

    print(
        f"Top-{n_features:>2} | "
        f"PR-AUC: {pr_auc:.4f} | "
        f"best iteration: {model.get_best_iteration()}"
    )

feature_selection_results = (
    pd.DataFrame(results)
    .sort_values("n_features")
    .reset_index(drop=True)
)

display(feature_selection_results[["n_features", "pr_auc", "best_iteration"]])

Top- 2 | PR-AUC: 0.3152 | best iteration: 0
Top- 4 | PR-AUC: 0.3324 | best iteration: 0
Top- 6 | PR-AUC: 0.3418 | best iteration: 0
Top- 8 | PR-AUC: 0.4002 | best iteration: 782
Top-10 | PR-AUC: 0.4035 | best iteration: 681
Top-12 | PR-AUC: 0.4113 | best iteration: 582
Top-14 | PR-AUC: 0.4133 | best iteration: 614
Top-16 | PR-AUC: 0.4604 | best iteration: 537
Top-18 | PR-AUC: 0.4629 | best iteration: 548


,n_features,pr_auc,best_iteration
0,2,0.315194,0
1,4,0.332367,0
2,6,0.341823,0
3,8,0.400192,782
4,10,0.403511,681
5,12,0.411285,582
6,14,0.413309,614
7,16,0.460416,537
8,18,0.462924,548


Видим, что даже с 16 фичами PR-AUC уже равен 0.46 на валидационном датасете (по сравнению с 0.465 со всеми фичами). То есть разница небольшая и можно оставить 16 фичей вообще практически без потери качества.

Но пока для упрощения процесса оставим только 10 фичей, которые дают PR-AUC 0.40.

## Финальные обучение и валидация на сокращенном числе фичей

In [27]:
BEST_FEATURES = ranked_features[:10]

X_train_final = X_train_cb[BEST_FEATURES]
X_val_final = X_val_cb[BEST_FEATURES]
X_test_final = X_test_cb[BEST_FEATURES]

final_cat_features = [
        feature
        for feature in categorical_features
        if feature in BEST_FEATURES
    ]

catboost_model = CatBoostClassifier(
    iterations=1_000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=100,
    od_type="Iter",
    od_wait=100,
    use_best_model=True,
    task_type="CPU",
    allow_writing_files=False,
)


catboost_model.fit(
    X_train_final,
    y_train,
    cat_features=final_cat_features,
    eval_set=(X_val_final, y_val),
)

0:	learn: 0.7857492	test: 0.7828219	best: 0.7828219 (0)	total: 75.4ms	remaining: 1m 15s
100:	learn: 0.8103815	test: 0.7956826	best: 0.7956826 (100)	total: 3.9s	remaining: 34.8s
200:	learn: 0.8216308	test: 0.7976631	best: 0.7980965 (181)	total: 7.4s	remaining: 29.4s
300:	learn: 0.8371459	test: 0.8007739	best: 0.8010694 (254)	total: 12.5s	remaining: 29s
400:	learn: 0.8500784	test: 0.8028170	best: 0.8030205 (393)	total: 16.9s	remaining: 25.2s
500:	learn: 0.8599031	test: 0.8043121	best: 0.8044561 (491)	total: 21s	remaining: 20.9s
600:	learn: 0.8679930	test: 0.8048231	best: 0.8050369 (567)	total: 26.3s	remaining: 17.4s
700:	learn: 0.8751887	test: 0.8048939	best: 0.8053153 (681)	total: 30.4s	remaining: 13s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8053153072
bestIteration = 681

Shrink model to first 682 iterations.


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, eval_metric='PRAUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', od_type='Iter', od_wait=100, random_seed=42, task_type='CPU', use_best_model=True, verbose=100)

In [25]:
val_metrics_05, val_proba, val_pred_05 = evaluate_classifier(
    model=catboost_model,
    X_data=X_val_final,
    y_true=y_val,
    threshold=0.5,
    dataset_name="validation"
    )

,dataset,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,validation,0.5,0.7988,0.4035,0.8432,0.3795,0.6173,0.47


Classification report:
              precision    recall  f1-score   support

           0     0.9472    0.8719    0.9080      5848
           1     0.3795    0.6173    0.4700       742

    accuracy                         0.8432      6590
   macro avg     0.6633    0.7446    0.6890      6590
weighted avg     0.8833    0.8432    0.8587      6590

Confusion matrix:
[[5099  749]
 [ 284  458]]


In [26]:
thresholds = np.arange(0.05, 0.96, 0.01)

results = []

for threshold in thresholds:
    predictions = (val_proba >= threshold).astype(int)

    results.append(
        {
            "threshold": float(threshold),
            "precision": precision_score(y_val, predictions, zero_division=0),
            "recall": recall_score(y_val, predictions, zero_division=0),
            "f1": f1_score(y_val, predictions, zero_division=0),
        }
    )

threshold_df = pd.DataFrame(results)

best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
BEST_THRESHOLD = float(best_threshold_row["threshold"])

print(f"Лучший threshold по F1: {BEST_THRESHOLD:.2f}")
display(pd.DataFrame([best_threshold_row]).round(4))

print("\nТоп-10 порогов по F1:")
display(threshold_df.sort_values("f1", ascending=False).head(10))

Лучший threshold по F1: 0.74


,threshold,precision,recall,f1
69,0.74,0.4622,0.5526,0.5034



Топ-10 порогов по F1:


,threshold,precision,recall,f1
69,0.74,0.462232,0.552561,0.503376
68,0.73,0.456763,0.555256,0.501217
66,0.71,0.451404,0.563342,0.501199
70,0.75,0.465817,0.541779,0.500935
64,0.69,0.445378,0.571429,0.500590
57,0.62,0.434093,0.590296,0.500286
67,0.72,0.452070,0.559299,0.500000
65,0.70,0.447761,0.566038,0.500000
56,0.61,0.432087,0.591644,0.499431
63,0.68,0.443051,0.571429,0.499117


Лучшая граница немного изменилась. Была 0.71, стала 0.74.

In [28]:
test_metrics, test_proba, test_pred = evaluate_classifier(
    model=catboost_model,
    X_data=X_test_final,
    y_true=y_test,
    threshold=BEST_THRESHOLD,
    dataset_name="test",
)

,dataset,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,test,0.74,0.8079,0.4276,0.881,0.4762,0.5614,0.5153


Classification report:
              precision    recall  f1-score   support

           0     0.9430    0.9216    0.9322      7310
           1     0.4762    0.5614    0.5153       928

    accuracy                         0.8810      8238
   macro avg     0.7096    0.7415    0.7238      8238
weighted avg     0.8904    0.8810    0.8852      8238

Confusion matrix:
[[6737  573]
 [ 407  521]]


In [29]:
print(test_metrics)

{'dataset': 'test', 'threshold': 0.7400000000000002, 'roc_auc': np.float64(0.8079386851974151), 'pr_auc': np.float64(0.42759549710473543), 'accuracy': 0.8810390871570769, 'precision': 0.47623400365630714, 'recall': 0.5614224137931034, 'f1': 0.5153313550939663}


Стоит заметить, что в данном случае F1 ухудшилась лишь совсем немного на тесте. Она равна **0.515** (**-2.5%** по сравнению с **0.5284** со всеми фичами)

## Оборачиваем модель в пайплайн

In [30]:
from dataclasses import dataclass
from typing import Any
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd


@dataclass
class CatBoostInferencePipeline:
    model: Any
    feature_names: list[str]
    categorical_features: list[str]
    numeric_features: list[str]
    numeric_medians: dict[str, float]

    def preprocess(self, X: pd.DataFrame | dict | list[dict]) -> pd.DataFrame:
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        elif isinstance(X, list):
            X = pd.DataFrame(X)
        else:
            X = X.copy()

        missing_features = set(self.feature_names) - set(X.columns)

        if missing_features:
            raise ValueError(
                f"Не хватает обязательных признаков: {sorted(missing_features)}"
            )

        # Удаляем лишние поля и гарантируем порядок, на котором обучалась модель
        X = X[self.feature_names].copy()

        for feature in self.categorical_features:
            X[feature] = X[feature].fillna("__MISSING__").astype(str)

        for feature in self.numeric_features:
            X[feature] = pd.to_numeric(X[feature], errors="coerce")
            X[feature] = X[feature].fillna(self.numeric_medians[feature])

        return X

    def predict_proba(self, X: pd.DataFrame | dict | list[dict]) -> np.ndarray:
        X_prepared = self.preprocess(X)
        return self.model.predict_proba(X_prepared)

    def predict(self, X: pd.DataFrame | dict | list[dict], threshold: float) -> np.ndarray:
        positive_proba = self.predict_proba(X)[:, 1]
        return (positive_proba >= threshold).astype(int)

## Грузим модель в правильный формат

Сначала загрузим сам пайплайн.

In [31]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

MODEL_VERSION = "0.1.0"

final_numeric_features = [
    feature
    for feature in numeric_features
    if feature in BEST_FEATURES
]

final_numeric_medians = {
    feature: float(numeric_medians[feature])
    for feature in final_numeric_features
}

final_cat_features = [
    feature
    for feature in categorical_features
    if feature in BEST_FEATURES
]

inference_pipeline = CatBoostInferencePipeline(
    model = catboost_model,
    feature_names = BEST_FEATURES,
    categorical_features = final_cat_features,
    numeric_features = final_numeric_features,
    numeric_medians = final_numeric_medians
)


Теперь загрузим метаданные

In [34]:
metadata = {
    "model_name": "bank_marketing_catboost",
    "model_version": MODEL_VERSION,
    "target_column": target,
    "positive_class": 1,
    "negative_class": 0,
    "threshold": float(BEST_THRESHOLD),
    "feature_names": BEST_FEATURES,
    "categorical_features": final_cat_features,
    "numeric_features": final_numeric_features,
    "numeric_medians": final_numeric_medians,
    "best_iteration": int(catboost_model.get_best_iteration()),
    "tree_count": int(catboost_model.tree_count_),
    "test_metrics": {
        key: float(value)
        for key, value in test_metrics.items()
        if key not in {"dataset", "threshold"}
    }
}

In [37]:
bundle = {
    "pipeline": catboost_model,
    "metadata": metadata
}

BUNDLE_PATH = ARTIFACT_DIR / "bank_marketing_model_catboost_bundle.joblib"

joblib.dump(bundle, BUNDLE_PATH, compress = 3)

['artifacts/bank_marketing_model_catboost_bundle.joblib']

In [38]:
print(f"Размер файла: {BUNDLE_PATH.stat().st_size / 1024:.1f} KB")

Размер файла: 475.4 KB


Проверим загрузку через инференс

In [39]:
loaded_bundle = joblib.load(BUNDLE_PATH)

loaded_pipeline = loaded_bundle["pipeline"]
loaded_metadata = loaded_bundle["metadata"]

input_example = X_test.loc[[X_test.index[4]], BEST_FEATURES]

In [42]:
X_test_final.loc

,euribor3m,nr.employed,month,campaign,job,education,age,emp.var.rate,cons.conf.idx,day_of_week
14455,4.961,5228.1,jul,5,management,university.degree,32,1.4,-42.7,tue
36380,1.262,5076.2,jun,1,unemployed,university.degree,37,-2.9,-40.8,tue
40076,0.810,4991.6,jul,2,retired,professional.course,73,-1.7,-40.3,thu
10778,4.961,5228.1,jun,2,entrepreneur,basic.4y,44,1.4,-41.8,tue
27939,1.531,5099.1,mar,2,admin.,high.school,28,-1.8,-50.0,fri


In [41]:
prob = loaded_pipeline.predict_proba(input_example)

res = loaded_pipeline.predict(
    input_example,
    threshold = loaded_metadata['threshold']
)

TypeError: CatBoostClassifier.predict() got an unexpected keyword argument 'threshold'

### Смотрим тип финальных выбранных фичей

In [ ]:
bundle = joblib.load('bank_marketing_catboost_bundle.joblib')

final_loaded_features = bundle['metadata']['feature_names']
print(f"Список имен признаков: {bundle['metadata']['feature_names']}")

df_final_loaded = df[final_loaded_features]

df_final_loaded.head(20)

Список имен признаков: ['euribor3m', 'nr.employed', 'month', 'campaign', 'job', 'education', 'age', 'emp.var.rate', 'cons.conf.idx', 'day_of_week']


,euribor3m,nr.employed,month,campaign,job,education,age,emp.var.rate,cons.conf.idx,day_of_week
0,4.857,5191.0,may,1,housemaid,basic.4y,56,1.1,-36.4,mon
1,4.857,5191.0,may,1,services,high.school,57,1.1,-36.4,mon
2,4.857,5191.0,may,1,services,high.school,37,1.1,-36.4,mon
3,4.857,5191.0,may,1,admin.,basic.6y,40,1.1,-36.4,mon
4,4.857,5191.0,may,1,services,high.school,56,1.1,-36.4,mon
5,4.857,5191.0,may,1,services,basic.9y,45,1.1,-36.4,mon
6,4.857,5191.0,may,1,admin.,professional.course,59,1.1,-36.4,mon
7,4.857,5191.0,may,1,blue-collar,unknown,41,1.1,-36.4,mon
8,4.857,5191.0,may,1,technician,professional.course,24,1.1,-36.4,mon
9,4.857,5191.0,may,1,services,high.school,25,1.1,-36.4,mon


In [ ]:
numeric_df = df_final_loaded.select_dtypes(include = np.number)

result = numeric_df.agg(["min", "max"])
print(result)

     euribor3m  nr.employed  campaign  age  emp.var.rate  cons.conf.idx
min      0.634       4963.6         1   17          -3.4          -50.8
max      5.045       5228.1        56   98           1.4          -26.9


In [ ]:
cat_df = df_final_loaded.select_dtypes(include=["object", "category", "string"])

for col in cat_df.columns:
    print(f"\n{'=' * 60}")
    print(f"Колонка: {col}")
    print(f"Число уникальных значений: {df[col].nunique()}")
    print(f"Уникальные значения: {df[col].unique()}")


Колонка: month
Число уникальных значений: 10
Уникальные значения: ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']

Колонка: job
Число уникальных значений: 12
Уникальные значения: ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']

Колонка: education
Число уникальных значений: 8
Уникальные значения: ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']

Колонка: day_of_week
Число уникальных значений: 5
Уникальные значения: ['mon' 'tue' 'wed' 'thu' 'fri']
